In [0]:
df_estrutura = spark.sql("""
    SELECT
        table_schema AS camada,
        table_name AS tabela,
        column_name AS coluna,
        data_type AS tipo
        --,ordinal_position AS ordem
    FROM olist_project.information_schema.columns
    WHERE table_schema IN (/*'bronze',*/ 'silver'/*, 'gold'*/)
    ORDER BY table_schema, table_name, ordinal_position
""")

display(df_estrutura)

# 📊 Dicionário de Dados — Camada Silver

Documentação das 9 tabelas da camada Silver do projeto Olist E-commerce.
Todas as tabelas (exceto indicado) possuem `_ingest_timestamp` e `_source_file` como metadados de auditoria.

---

## `silver.orders`
Tabela central — cada linha é um pedido.

| Campo | Descrição |
|---|---|
| `order_id` | Identificador único do pedido (chave primária) |
| `customer_id` | Referência ao cliente que fez o pedido |
| `order_status` | Status do pedido (delivered, shipped, canceled, etc.) |
| `order_purchase_timestamp` | Data/hora em que o pedido foi realizado |
| `order_approved_at` | Data/hora em que o pagamento foi aprovado |
| `order_delivered_carrier_date` | Data em que o pedido foi entregue à transportadora |
| `order_delivered_customer_date` | Data em que o pedido chegou ao cliente |
| `order_estimated_delivery_date` | Data estimada de entrega (prometida ao cliente) |
| `flag_data_inconsistente` | `True` quando a entrega aparece antes da compra |

---

## `silver.customers`
Um registro por cliente (por pedido — ver `customer_unique_id` pra identificar a pessoa real).

| Campo | Descrição |
|---|---|
| `customer_id` | Chave usada para ligar com `orders` |
| `customer_unique_id` | Identificador real da pessoa (permite ver recompra) |
| `customer_zip_code_prefix` | Prefixo do CEP |
| `customer_city` | Cidade (padronizada em minúsculo/trim) |
| `customer_state` | Estado, sigla (padronizado em maiúsculo/trim) |

---

## `silver.order_items`
Um registro por item dentro de um pedido.

| Campo | Descrição |
|---|---|
| `order_id` | Referência ao pedido |
| `order_item_id` | Número sequencial do item dentro do pedido |
| `product_id` | Referência ao produto |
| `seller_id` | Referência ao vendedor daquele item |
| `shipping_limit_date` | Prazo limite pro vendedor despachar o item |
| `price` | Preço do item |
| `freight_value` | Valor do frete daquele item |

---

## `silver.payments`
Um ou mais registros de pagamento por pedido.

| Campo | Descrição |
|---|---|
| `order_id` | Referência ao pedido |
| `payment_sequential` | Ordem do pagamento, se houve mais de um |
| `payment_type` | Forma de pagamento (credit_card, boleto, voucher, debit_card) |
| `payment_installments` | Número de parcelas |
| `payment_value` | Valor pago naquela transação |

---

## `silver.reviews`
Uma avaliação por pedido (nem todo pedido tem review).

| Campo | Descrição |
|---|---|
| `review_id` | Identificador da avaliação |
| `order_id` | Referência ao pedido avaliado |
| `review_score` | Nota de 1 a 5 |
| `review_comment_title` | Título do comentário (texto livre, muitos nulos) |
| `review_comment_message` | Corpo do comentário (texto livre, muitos nulos) |
| `review_creation_date` | Data em que a avaliação foi criada |
| `review_answer_timestamp` | Data em que a avaliação foi respondida |

---

## `silver.products`
Um registro por produto.

| Campo | Descrição |
|---|---|
| `product_id` | Identificador do produto |
| `product_category_name` | Categoria (em português; nulos preenchidos como "nao_informado") |
| `product_name_lenght` | Quantidade de caracteres no nome do produto |
| `product_description_lenght` | Quantidade de caracteres na descrição do produto |
| `product_photos_qty` | Quantidade de fotos cadastradas pro produto |
| `product_weight_g` | Peso em gramas |
| `product_length_cm` | Comprimento em cm |
| `product_height_cm` | Altura em cm |
| `product_width_cm` | Largura em cm |

---

## `silver.sellers`
Um registro por vendedor.

| Campo | Descrição |
|---|---|
| `seller_id` | Identificador do vendedor |
| `seller_zip_code_prefix` | Prefixo do CEP |
| `seller_city` | Cidade (padronizado) |
| `seller_state` | Estado, sigla (padronizado) |

---

## `silver.geolocation` 🆕
Um registro por prefixo de CEP (agregado a partir da Bronze, que tinha múltiplas linhas por CEP).

| Campo | Descrição |
|---|---|
| `geolocation_zip_code_prefix` | Prefixo do CEP (chave, agora única) |
| `latitude` | Latitude média das ocorrências daquele CEP |
| `longitude` | Longitude média das ocorrências daquele CEP |
| `cidade` | Cidade mais frequente associada ao CEP |
| `estado` | Estado, sigla, associado ao CEP |

---

## `silver.category_translation` 🆕
Tabela de tradução — de-para entre nome de categoria em português e inglês.

| Campo | Descrição |
|---|---|
| `product_category_name` | Nome da categoria em português (chave, usada pra ligar com `products`) |
| `product_category_name_english` | Nome equivalente da categoria em inglês |